# 🔄 Notebook 02 — Delta MERGE: Upserts & Late-Arriving Data

**Goal:** Use Delta's `MERGE` command to handle upserts, corrections, and late-arriving data without full reloads.

> **Run time:** ~5 min

## Why MERGE?
```
INSERT overwrites — loses history
APPEND creates duplicates — wrong totals
MERGE handles all cases:
  WHEN MATCHED AND changed   → UPDATE
  WHEN MATCHED AND same      → skip (no-op)
  WHEN NOT MATCHED           → INSERT
  WHEN NOT MATCHED BY SOURCE → DELETE (optional)
```

## Banking use case: Loan status corrections
A loan's status changes from `Current` → `30-Days Late`. The core banking system sends a correction. We need to update the existing record — not create a duplicate.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Show current loan status distribution
print('Current fact_loans status distribution:')
spark.sql("SELECT LoanStatus, COUNT(*) AS Count FROM fact_loans GROUP BY LoanStatus ORDER BY Count DESC").show()

## Step 1 — Simulate Incoming Corrections

In [ ]:
# Simulate a batch of corrections from the core banking system:
# - Some existing loans changed status
# - Some outstanding balances updated
# - A few new loans not in the original load
corrections_data = [
    # (LoanID,   AccountID,   CustomerID,  ProductID,   BranchID, LoanAmount, InterestRate, TermMonths, MonthlyPayment, OutstandingBalance, StartDate,    EndDate,      LoanStatus,       LoanPurpose)
    ('LOAN0001', 'ACC0001', 'CUST0001', 'PROD001', 'BR001', 15000.0, 8.5,  36, 472.0, 8200.0,  '2022-03-01', '2025-03-01', '30-Days Late',    'Personal'),    # status change
    ('LOAN0002', 'ACC0002', 'CUST0002', 'PROD002', 'BR002', 250000.0,4.2, 360, 1223.0,238000.0,'2021-06-01', '2051-06-01', 'Current',         'Home Purchase'),# balance update
    ('LOAN0050', 'ACC0050', 'CUST0050', 'PROD003', 'BR005', 22000.0, 6.9,  60, 433.0, 15000.0, '2023-01-01', '2028-01-01', 'Paid Off',        'Auto Purchase'),# status change
    ('LOAN0099', 'ACC0099', 'CUST0099', 'PROD005', 'BR003', 180000.0,10.5, 60, 3858.0,160000.0,'2024-11-01', '2029-11-01', 'Current',         'Business Expansion'),# balance update
    ('LOAN0501', 'ACC0150', 'CUST0150', 'PROD001', 'BR007', 12000.0, 9.2,  48, 299.0, 12000.0, '2025-02-01', '2029-02-01', 'Current',         'Personal'),    # NEW loan
    ('LOAN0502', 'ACC0250', 'CUST0250', 'PROD004', 'BR008', 45000.0, 5.1, 120, 478.0, 45000.0, '2025-02-15', '2035-02-15', 'Current',         'Education'),   # NEW loan
]

columns = ['LoanID','AccountID','CustomerID','ProductID','BranchID','LoanAmount','InterestRate',
           'TermMonths','MonthlyPayment','OutstandingBalance','StartDate','EndDate','LoanStatus','LoanPurpose']

df_corrections = spark.createDataFrame(corrections_data, columns) \
    .withColumn('_corrected_at', F.current_timestamp())

print(f'Incoming corrections: {df_corrections.count()} records')
df_corrections.select('LoanID','LoanStatus','OutstandingBalance','LoanPurpose').show()

## Step 2 — MERGE into fact_loans

In [ ]:
target = DeltaTable.forName(spark, 'fact_loans')

target.alias('target').merge(
    df_corrections.alias('source'),
    'target.LoanID = source.LoanID'
) \
.whenMatchedUpdate(set={
    'LoanStatus':         'source.LoanStatus',
    'OutstandingBalance': 'source.OutstandingBalance',
}) \
.whenNotMatchedInsert(values={
    'LoanID':             'source.LoanID',
    'AccountID':          'source.AccountID',
    'CustomerID':         'source.CustomerID',
    'ProductID':          'source.ProductID',
    'BranchID':           'source.BranchID',
    'LoanAmount':         'source.LoanAmount',
    'InterestRate':       'source.InterestRate',
    'TermMonths':         'source.TermMonths',
    'MonthlyPayment':     'source.MonthlyPayment',
    'OutstandingBalance': 'source.OutstandingBalance',
    'StartDate':          'source.StartDate',
    'EndDate':            'source.EndDate',
    'LoanStatus':         'source.LoanStatus',
    'LoanPurpose':        'source.LoanPurpose',
}) \
.execute()

print('✅ MERGE complete')

## Step 3 — Verify Results

In [ ]:
%%sql
-- Confirm updates and inserts
SELECT LoanID, LoanStatus, OutstandingBalance, LoanPurpose
FROM fact_loans
WHERE LoanID IN ('LOAN0001','LOAN0002','LOAN0050','LOAN0099','LOAN0501','LOAN0502')
ORDER BY LoanID

In [ ]:
%%sql
-- Confirm no duplicates (LoanID should be unique)
SELECT COUNT(*) AS TotalLoans, COUNT(DISTINCT LoanID) AS UniqueLoanIDs
FROM fact_loans

## Step 4 — View Delta MERGE Operation Log

In [ ]:
# View Delta transaction log — shows MERGE stats
history = DeltaTable.forName(spark, 'fact_loans').history(3)
history.select('version','timestamp','operation','operationMetrics').show(truncate=False)

## Step 5 — SCD Type 2 Preview (Slowly Changing Dimensions)

For dimension tables where you want to keep full history (e.g. a customer changed their segment):

In [ ]:
%%sql
-- SCD Type 2 pattern: expire old record, insert new one
-- This shows the concept — in production you'd add EffectiveDate/ExpiryDate columns
MERGE INTO dim_customer AS target
USING (
    SELECT 'CUST0001' AS CustomerID, 'Corporate' AS CustomerSegment, current_date() AS EffectiveDate
) AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED AND target.CustomerSegment != source.CustomerSegment
    THEN UPDATE SET target.CustomerSegment = source.CustomerSegment,
                    target._updated_at = current_timestamp()
-- WHEN NOT MATCHED: insert new version (full SCD Type 2 requires EffectiveDate/ExpiryDate columns)